# 07 · E2: train the answering adapter (Step 3.2)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**).

1. Quick check that vLLM can load a LoRA adapter (uses the smoke-test adapter from notebook 05, if it is still on Drive).
2. **Train** the answering adapter: bf16 LoRA, 13,134 training questions, 1 epoch (821 optimizer steps), image budget 524,288 px (D-050). Checkpoint every 10 minutes to Drive; the curve is logged to W&B.
3. **Evaluate** it on the full test set (2,934 answerable questions), same pipeline as E1.
4. **Compare** with E1 question by question (paired bootstrap).

Expect about **3–3.5 hours**, almost all of it training. Keep the tab open.

**If Colab disconnects:** reconnect (L4) and **Run all** again. Training resumes from the last checkpoint (at most 10 minutes lost); finished steps are skipped.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code, W&B key

In [ ]:
import json
import os
import subprocess

from google.colab import drive, userdata

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
CKPT = f"{DRIVE_ROOT}/checkpoints"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
RUN_ID = "e2-answer-lora"
ADAPTER = f"{CKPT}/{RUN_ID}/adapter"


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, and raise if it fails (unless check=False)."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return code


try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")  # never printed
    print("W&B key loaded from Colab Secrets.")
except Exception as e:
    raise RuntimeError(
        "Add WANDB_API_KEY in the key icon (Secrets) on the left, turn on notebook access, then rerun."
    ) from e
os.environ["WANDB_PROJECT"] = "vizwiz-assistant"
os.environ["HF_HOME"] = "/content/hf_cache"

drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

**Paste back:** the commit line and the GPU line (should say NVIDIA L4).

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Unzip photos (~4 min)
Training and val photos come from `train.zip`; the test set (official val) from `val.zip`. Label files are checked against the committed hashes by the training script and below.

In [ ]:
%%time
import hashlib

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_test.jsonl", "quality_test.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits train val")

## 4 · Can vLLM load a LoRA adapter? (~4 min, first model download here)
Evaluates the 30-step smoke adapter from notebook 05 on 64 val questions. The score does not matter; this only proves the adapter format works with vLLM **before** 3 hours of training. Skipped if you deleted the smoke checkpoints.

In [ ]:
%%time
SMOKE_ADAPTER = f"{CKPT}/smoke-lora/adapter"
if os.path.exists(f"{SMOKE_ADAPTER}/adapter_config.json"):
    sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split val --tasks answer --limit 64 --adapter {SMOKE_ADAPTER} --run-id adapter-check --out /content/adapter-check"
    )
    print("ADAPTER LOAD OK")
else:
    print(
        "smoke adapter not on Drive: check skipped (the evaluation in cell 6 will test it instead)"
    )

**Paste back:** `ADAPTER LOAD OK` (or the skip line), or the error if it failed. **If it failed, stop here** and paste the error: training would still work, but the adapter could not be served yet.

## 5 · Train (~2.5–3 h)
Bf16 LoRA on the language model, 821 optimizer steps (batch 1 × 16 accumulation). The val loss is computed on 500 val questions every 200 steps. Progress prints as it goes; the live curve is in W&B (project `vizwiz-assistant`, run `e2-answer-lora`).

After a disconnect, this same cell resumes from the newest checkpoint. If training already finished, it is skipped.

In [ ]:
%%time
summary_path = f"{CKPT}/{RUN_ID}/train_summary.json"
if os.path.exists(summary_path) and os.path.exists(f"{ADAPTER}/adapter_config.json"):
    print("training already finished: skipped")
else:
    sh(
        f"{PY} -m vizwiz.train.sft train --config {REPO_DIR}/configs/train/answer.yaml --mode lora --records {DATA}/processed --images {LOCAL}/images --out {CKPT} --run-id {RUN_ID} --wandb --wandb-group e2"
    )
s = json.load(open(summary_path))
print(
    "TRAIN DONE:",
    {
        k: s[k]
        for k in (
            "global_step",
            "optimizer_steps_planned",
            "resumed_from",
            "train_loss",
            "final_val_loss",
            "examples_per_s",
            "peak_mem_gb",
            "wall_s",
        )
    },
)

**Paste back:** the `TRAIN DONE: ...` line and the W&B run link (printed near the start, `wandb: 🚀 View run ...`).

## 6 · Evaluate on the full test set (~8 min)
Same command as E1 plus `--adapter`. Skipped if already done.

In [ ]:
%%time
OUT = f"{RUNS}/{RUN_ID}"
done = (
    os.path.exists(f"{OUT}/metrics.json")
    and "answer" in json.load(open(f"{OUT}/metrics.json"))["tasks"]
)
if done:
    print("evaluation already done: skipped")
else:
    sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split test --tasks answer --adapter {ADAPTER} --run-id {RUN_ID} --out {OUT}"
    )

**Paste back:** the `[answer] vqa_accuracy = ...` line.

## 7 · Compare with E1, question by question (~1 min)

In [ ]:
sh(
    f"{PY} -m vizwiz.eval.compare --records {DATA}/processed --split test --task answer --runs {RUNS}/e1-baseline {OUT} --names e1 e2 --reference e1 --report {OUT}/compare_answer.json > /dev/null"
)
c = json.load(open(f"{OUT}/compare_answer.json"))
print(
    "COMPARE:",
    c["n"],
    {k: (r["value"], r["diff_vs_reference"], r["diff_ci95"]) for k, r in c["runs"].items()},
)
e1 = json.load(open(f"{RUNS}/e1-baseline/metrics.json"))["tasks"]["answer"]["metrics"]
e2 = json.load(open(f"{OUT}/metrics.json"))["tasks"]["answer"]["metrics"]
for key in ("by_answer_type", "by_quality_any_issue"):
    for g in e2[key]:
        print(
            f"{key:22s} {g:12s} n={e2[key][g]['n']:5d}  e1 {e1[key][g]['accuracy']:6.2f}  e2 {e2[key][g]['accuracy']:6.2f}"
        )
print("declined %: e1", e1["declined_pct"], " e2", e2["declined_pct"])

**Paste back:** everything this cell printed (numbers only).

## 8 · Flush Drive, then disconnect
Upload back (summary numbers only; prediction files and checkpoints stay on Drive):
- `MyDrive/vizwiz/checkpoints/e2-answer-lora/train_summary.json`
- `MyDrive/vizwiz/checkpoints/e2-answer-lora/settings.json`
- `MyDrive/vizwiz/runs/e2-answer-lora/metrics.json`
- `MyDrive/vizwiz/runs/e2-answer-lora/compare_answer.json`

Then Runtime → **Disconnect and delete runtime** (an idle L4 keeps using compute units).

Keep `checkpoints/e2-answer-lora/adapter/` on Drive: it is the trained adapter (~130 MB). The two `checkpoint-*` folders next to it (~0.8 GB) can be deleted once the numbers look right.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")